# Y-mas Tier 2 — YOLO11n-pose ONNX Export

AGX Orin 실시간 추론(`edge/ymas_realtime_ir.py`)이 사용하는 **YOLO11n-pose ONNX** 모델을
Colab(무료 T4)에서 만들어 Drive로 내보낸다.

> **왜 Colab에서?** Orin에서 `pip install ultralytics` 하면 범용 torch/CUDA13 뎍치가 딜려와
> JetPack 환경(CUDA 12.6)을 오염하고 eMMC를 치면한다. export는 GPU 성능과 무관하므로
> 무료 T4로 충분하며, 결과 `.onnx` 파일만 Orin `~/Y-mas/edge/` 로 옳긴다.

## 불변식 (중요)
- **`imgsz=320`** — 실시간 코드의 `YOLO_SIZE=320` 과 반드시 일치해야 함.
- **`opset=12`** — Orin TensorRT 10.3 / onnxruntime 파싱 호환.
- 출력 레이아웃은 `[1, 56, 8400]` (56 = 4 bbox + 1 conf + 51 kpt(17×3)).
  실시간 코드가 `output[0].T` → `preds[:,4]`(conf) → `det[5:].reshape(17,3)` 로 파싱함.

## 1. ultralytics 설치 (Colab 전용)

In [ ]:
!pip install -q ultralytics


## 2. YOLO11n-pose → ONNX export

`yolo11n-pose.pt` (pretrained) 를 자동 다운로드해 ONNX 로 내보낸다.

In [ ]:
from ultralytics import YOLO

m = YOLO("yolo11n-pose.pt")            # pretrained 자동 다운로드
path = m.export(format="onnx", imgsz=320, opset=12)
print("export →", path)
!ls -lh yolo11n-pose.onnx


## 3. (선택) 출력 shape 검증

실시간 코드 파싱과 맞는지 확인: 출력이 `(1, 56, 8400)` 이면 OK.

In [ ]:
import onnxruntime as ort, numpy as np

sess = ort.InferenceSession("yolo11n-pose.onnx", providers=["CPUExecutionProvider"])
inp = sess.get_inputs()[0]
print("input :", inp.name, inp.shape)
dummy = np.zeros((1, 3, 320, 320), dtype=np.float32)
out = sess.run(None, {inp.name: dummy})[0]
print("output:", out.shape, "(기대: (1, 56, 8400))")


## 4. Drive 저장 (Orin Firefox 로 다운로드)

Drive에 저장 후, Orin 에서 Firefox 로 받아 `~/Y-mas/edge/yolo11n-pose.onnx` 로 둔다.
실시간 코드가 해당 경로를 자동으로 찾는다 (`YMAS_POSE_ONNX` 로 override 가능).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp yolo11n-pose.onnx /content/drive/MyDrive/
print("Drive 저장 완료 → Orin 에서 다운로드 후 ~/Y-mas/edge/ 로 이동")
